# SCRAPING IHSG

In [11]:
# ============================================================
# 2.1 DOWNLOAD DATA IHSG DARI YAHOO FINANCE
# ============================================================

# Install library (skip kalau sudah)
# !pip install yfinance pandas numpy

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
import os

# Buat folder raw kalau belum ada
os.makedirs('../data/raw', exist_ok=True)

# Download data IHSG
print("⏳ Mendownload data IHSG dari Yahoo Finance...")
ticker = yf.Ticker("^JKSE")

# Ambil data harian 2019-2024
df = ticker.history(start="2019-01-01", end="2024-12-31")

print(f"✅ Berhasil download! Total baris: {len(df)}")

# Lihat 5 baris pertama
display(df.head())

# ============================================================
# HITUNG DERIVED METRICS
# ============================================================

# Reset index agar Date jadi kolom biasa
df = df.reset_index()

# Rename kolom (hapus spasi/camelCase)
df.columns = ['date', 'open', 'high', 'low', 'close', 'volume', 'dividends', 'stock_splits']

# Hapus kolom yang tidak perlu (IHSG tidak ada dividen/stock split)
df = df.drop(columns=['dividends', 'stock_splits'])

# Daily Return
df['daily_return'] = df['close'].pct_change()

# Log Return
df['log_return'] = np.log(df['close'] / df['close'].shift(1))

# Volatilitas 20 hari (annualized)
df['volatility_20d'] = df['log_return'].rolling(window=20).std() * np.sqrt(252)

# Moving Average 50 hari
df['ma_50'] = df['close'].rolling(window=50).mean()

# Format tanggal (hapus timezone)
df['date'] = pd.to_datetime(df['date']).dt.tz_localize(None)

print("\n📊 Preview setelah engineering:")
display(df.tail())

# ============================================================
# SIMPAN KE CSV
# ============================================================

output_path = r'C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\raw\ihsg_2019_2024.csv'
df.to_csv(output_path, index=False)
print(f"\n💾 Data disimpan di: {output_path}")
print(f"📁 Ukuran file: {os.path.getsize(output_path) / 1024:.1f} KB")

⏳ Mendownload data IHSG dari Yahoo Finance...
✅ Berhasil download! Total baris: 1456


,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2019-01-02 00:00:00+07:00,6197.871094,6205.895020,6164.833984,6181.174805,52797800,0.0,0.0
2019-01-03 00:00:00+07:00,6176.151855,6221.009766,6176.151855,6221.009766,72166700,0.0,0.0
2019-01-04 00:00:00+07:00,6211.096191,6274.540039,6200.854004,6274.540039,80858100,0.0,0.0
2019-01-07 00:00:00+07:00,6317.625977,6354.757812,6287.224121,6287.224121,90278300,0.0,0.0
2019-01-08 00:00:00+07:00,6292.263184,6316.240234,6251.375977,6262.847168,90537400,0.0,0.0



📊 Preview setelah engineering:


,date,open,high,low,close,volume,daily_return,log_return,volatility_20d,ma_50
1451,2024-12-20,6980.174805,7032.400879,6931.581055,6983.865234,145749700,0.000950,0.000949,0.179546,7384.169434
1452,2024-12-23,7037.529785,7096.444824,7035.727051,7096.444824,138664700,0.016120,0.015991,0.187238,7375.686289
1453,2024-12-24,7115.637207,7120.576172,7063.754883,7065.746094,110632900,-0.004326,-0.004335,0.176358,7365.808115
1454,2024-12-27,7073.375000,7100.270020,7024.714844,7036.570801,144277600,-0.004129,-0.004138,0.174316,7354.000508
1455,2024-12-30,7026.776855,7079.904785,6993.071777,7079.904785,159672400,0.006158,0.006140,0.175317,7342.619805



💾 Data disimpan di: C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\raw\ihsg_2019_2024.csv
📁 Ukuran file: 234.9 KB


In [12]:
# Cek missing values
print("Missing values per kolom:")
print(df.isnull().sum())

# Cek rentang tanggal
print(f"\nRentang tanggal: {df['date'].min()} sampai {df['date'].max()}")
print(f"Total hari perdagangan: {len(df)}")

Missing values per kolom:
date               0
open               0
high               0
low                0
close              0
volume             0
daily_return       1
log_return         1
volatility_20d    20
ma_50             49
dtype: int64

Rentang tanggal: 2019-01-02 00:00:00 sampai 2024-12-30 00:00:00
Total hari perdagangan: 1456


# SCRAPING BERITA

In [13]:
# ============================================================
# GOOGLE NEWS RSS - HISTORICAL (2019–2024)
# ============================================================

import feedparser
import pandas as pd
import urllib.parse
import time

def get_google_news_rss_historical(query, start_date="2019-01-01", end_date="2024-12-31", max_results=100):
    """
    Ambil berita historis dari Google News RSS memanfaatkan operator query 'after:' dan 'before:'.
    """
    articles = []
    
    # Menambahkan operator rentang tanggal pada query Google
    full_query = f"{query} after:{start_date} before:{end_date}"
    
    # Encode query untuk URL
    q_encoded = urllib.parse.quote(full_query)
    
    # RSS endpoint Google News (Bahasa Indonesia)
    rss_url = f"https://news.google.com/rss/search?q={q_encoded}&hl=id&gl=ID&ceid=ID:id"
    
    print(f"⏳ Mengambil: {full_query}")
    feed = feedparser.parse(rss_url)
    
    print(f"   Ditemukan: {len(feed.entries)} entri")
    
    for entry in feed.entries[:max_results]:
        try:
            # Parse tanggal
            if hasattr(entry, 'published_parsed') and entry.published_parsed:
                date_obj = pd.Timestamp(*entry.published_parsed[:6])
                date_str = date_obj.strftime("%Y-%m-%d")
            else:
                date_str = None
            
            # Ambil sumber
            source = 'Google News'
            if hasattr(entry, 'source') and hasattr(entry.source, 'title'):
                source = entry.source.title
            
            articles.append({
                'title': entry.title,
                'url': entry.link,
                'publish_date': date_str,
                'source': source,
                'category': 'Ekonomi'
            })
            
        except Exception as e:
            continue
    
    return pd.DataFrame(articles)


# ============================================================
# JALANKAN PER TAHUN / PER QUARTER UNTUK HASIL MAKSIMAL
# ============================================================

queries = [
    "IHSG indeks harga saham gabungan",
    "saham Indonesia bursa efek",
    "ekonomi Indonesia inflasi",
    "BI Rate suku bunga Bank Indonesia",
    "rupiah melemah menguat"
]

# Google RSS membatasi max 100 item per request,
# Memecah per tahun (2019-2024) akan menghasilkan jauh lebih banyak artikel historis.
years = [2019, 2020, 2021, 2022, 2023, 2024]

all_articles = []

for year in years:
    start_d = f"{year}-01-01"
    end_d = f"{year}-12-31"
    
    for q in queries:
        df_temp = get_google_news_rss_historical(q, start_date=start_d, end_date=end_d, max_results=100)
        if len(df_temp) > 0:
            all_articles.append(df_temp)
        time.sleep(1.5)

# Gabungkan seluruh data
if all_articles:
    df_news = pd.concat(all_articles, ignore_index=True)
    
    # Hapus duplikat berdasarkan judul
    df_news = df_news.drop_duplicates(subset=['title'], keep='first')

    print(f"\n✅ TOTAL BERITA UNIK HISTORIS (2019–2024): {len(df_news)}")
    print(df_news.head(10))
    print(f"\nRentang Tanggal: {df_news['publish_date'].min()} s.d. {df_news['publish_date'].max()}")
    print(f"\nPer source:\n{df_news['source'].value_counts()}")

    # Simpan
    df_news.to_csv(r'C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\raw\news_googlerss_2019_2024.csv', index=False)
    print("\n💾 Disimpan ke: news_all_merged_2019_2024.csv")
else:
    print("\n⚠️ Tidak ada data yang berhasil ditarik.")

⏳ Mengambil: IHSG indeks harga saham gabungan after:2019-01-01 before:2019-12-31
   Ditemukan: 47 entri
⏳ Mengambil: saham Indonesia bursa efek after:2019-01-01 before:2019-12-31
   Ditemukan: 80 entri
⏳ Mengambil: ekonomi Indonesia inflasi after:2019-01-01 before:2019-12-31
   Ditemukan: 48 entri
⏳ Mengambil: BI Rate suku bunga Bank Indonesia after:2019-01-01 before:2019-12-31
   Ditemukan: 49 entri
⏳ Mengambil: rupiah melemah menguat after:2019-01-01 before:2019-12-31
   Ditemukan: 43 entri
⏳ Mengambil: IHSG indeks harga saham gabungan after:2020-01-01 before:2020-12-31
   Ditemukan: 46 entri
⏳ Mengambil: saham Indonesia bursa efek after:2020-01-01 before:2020-12-31
   Ditemukan: 91 entri
⏳ Mengambil: ekonomi Indonesia inflasi after:2020-01-01 before:2020-12-31
   Ditemukan: 47 entri
⏳ Mengambil: BI Rate suku bunga Bank Indonesia after:2020-01-01 before:2020-12-31
   Ditemukan: 48 entri
⏳ Mengambil: rupiah melemah menguat after:2020-01-01 before:2020-12-31
   Ditemukan: 41 entri
⏳ Me

In [14]:
# ============================================================
# CELL 9: VERIFIKASI AKHIR FASE 2
# ============================================================

print("=" * 60)
print("📊 VERIFIKASI DATASET FASE 2")
print("=" * 60)

# Cek IHSG
ihsg = pd.read_csv(r'C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\raw\ihsg_2019_2024.csv')
print(f"\n📈 IHSG:")
print(f"   Baris: {len(ihsg)}")
print(f"   Kolom: {list(ihsg.columns)}")
print(f"   Tanggal: {ihsg['date'].min()} → {ihsg['date'].max()}")

# Cek Berita
news = pd.read_csv(r'C:\Users\ADVAN\Desktop\analisis_sentimen_ihsg\data\raw\news_googlerss_2019_2024.csv')
print(f"\n📰 BERITA:")
print(f"   Baris: {len(news)}")
print(f"   Kolom: {list(news.columns)}")
print(f"   Per source:\n{news['source'].value_counts()}")

print(f"\n{'=' * 60}")
print("✅ FASE 2 SELESAI kalau:")

checks = [
    (len(ihsg) > 1000, f"IHSG > 1000 baris (aktual: {len(ihsg)})"),
    ('daily_return' in ihsg.columns, "Ada kolom daily_return"),
    (len(news) > 100, f"Berita > 100 artikel (aktual: {len(news)})"),
    ('publish_date' in news.columns, "Ada kolom publish_date"),
]

for ok, msg in checks:
    status = "✅" if ok else "❌"
    print(f"   {status} {msg}")

print("=" * 60)

📊 VERIFIKASI DATASET FASE 2

📈 IHSG:
   Baris: 1456
   Kolom: ['date', 'open', 'high', 'low', 'close', 'volume', 'daily_return', 'log_return', 'volatility_20d', 'ma_50']
   Tanggal: 2019-01-02 → 2024-12-30

📰 BERITA:
   Baris: 1905
   Kolom: ['title', 'url', 'publish_date', 'source', 'category']
   Per source:
source
CNBC Indonesia           374
kompas.id                 99
kontan.co.id              88
detikFinance              64
Liputan6.com              61
                        ... 
Fajarpos Network           1
indoposco.id               1
PelitaRiau.Com             1
Indonesia Investments      1
Rmol.id                    1
Name: count, Length: 211, dtype: int64

✅ FASE 2 SELESAI kalau:
   ✅ IHSG > 1000 baris (aktual: 1456)
   ✅ Ada kolom daily_return
   ✅ Berita > 100 artikel (aktual: 1905)
   ✅ Ada kolom publish_date
